# SIM Intervention Demand Forecasting — Model Training

## 5.1. Objective

The objective of this stage is to train and evaluate forecasting models for daily SIM intervention counts across six operational divisions.

We will train LightGBM, CatBoost, and Linear Regression using the 12 candidate predictive features developed in the Feature Engineering stage. After validating these models, we will build and evaluate an Ensemble that combines predictions from selected models. The Ensemble method and its combination weights will be defined and justified before its evaluation.

Model training and evaluation will follow these requirements:

- Use monthly expanding-window validation from January 2021 through December 2024.
- Respect the D−3 historical information cutoff.
- Keep each fitted model fixed within its validation month.
- Evaluate forecasts using Mean Absolute Error (MAE) and prediction bias.
- Use the selected 13-observation same-weekday baseline as the reference method.
- Keep the 2025–2026 final test period reserved.
- Avoid using a validation month's actual outcomes to determine the Ensemble weights used to forecast that same month.

This stage will generate and evaluate predictions for LightGBM, CatBoost, Linear Regression and the Ensemble. The next stage, Model Comparison and Selection, will compare all four approaches with the historical baseline and make a provisional model selection.


## 5.2. Install and Import Required Libraries

We will install LightGBM and import the Python libraries required for data preparation, model training, and evaluation.

We will use pandas and NumPy to work with the datasets, LightGBM to train the first forecasting model and scikit-learn to calculate evaluation metrics.

Additional libraries for CatBoost and Linear Regression will be introduced in their respective sections. We will also review whether the Ensemble requires any additional libraries when we implement it.

In [ ]:

# Install LightGBM
%pip install lightgbm


/home/pliqui/.cache/uv/builds-v0/.tmpz6VwEt/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:

# Import libraries
import pandas as pd
import numpy as np

import lightgbm as lgb

from sklearn.metrics import mean_absolute_error

# Verify the LightGBM installation
print("LightGBM version:", lgb.__version__)


LightGBM version: 4.7.0


## 5.3. Load Modeling Data and Validation Schedule

We will load the supervised modeling dataset and the chronological validation schedule generated in previous stages.

The modeling dataset provides the predictive features and known intervention counts. The validation schedule defines the monthly training cutoffs and evaluation periods.

We will verify both inputs before training the first LightGBM model.

In [ ]:

# Define file paths
modeling_file = "../../data/interim/final/SIM_Features_Modeling.csv"
schedule_file = "../../data/interim/final/SIM_Chronological_Validation_Schedule.csv"

# Load the modeling dataset
df_modeling = pd.read_csv(
    modeling_file,
    parse_dates=["Date"]
)

# Load the chronological validation schedule
df_schedule = pd.read_csv(
    schedule_file,
    parse_dates=[
        "Validation_Start",
        "Validation_End",
        "Training_Cutoff"
    ]
)

# Sort modeling observations chronologically
df_modeling = df_modeling.sort_values(
    ["Date", "DIVISION"]
).reset_index(drop=True)

# Verify the datasets
print("Modeling dataset dimensions:", df_modeling.shape)
print("Validation schedule dimensions:", df_schedule.shape)

print("Modeling start date:", df_modeling["Date"].min().date())
print("Modeling end date:", df_modeling["Date"].max().date())

print("Validation months:", len(df_schedule))

# Preview the validation schedule
display(df_schedule.head())


Modeling dataset dimensions: (14748, 14)
Validation schedule dimensions: (48, 5)
Modeling start date: 2020-01-01
Modeling end date: 2026-09-24
Validation months: 48


,Validation_Start,Validation_End,Training_Cutoff,Training_Rows,Validation_Rows
0,2021-01-01,2021-01-31,2020-12-29,2184,186
1,2021-02-01,2021-02-28,2021-01-29,2370,168
2,2021-03-01,2021-03-31,2021-02-26,2538,186
3,2021-04-01,2021-04-30,2021-03-29,2724,180
4,2021-05-01,2021-05-31,2021-04-28,2904,186


## 5.4. Define Predictive Features and Target

We will identify the input features (X) and target variable (y) used to train the machine learning models.

The models will use 12 candidate predictive features: the operational division, five calendar features, four lag features, and two rolling average features.

The target variable is `Intervention_Count`, representing the recorded daily interventions for each operational division.

The `Date` column will be used to control chronological training and validation periods, but it will not be included directly as a predictive feature.

In [ ]:

# Define the predictive features
feature_columns = [
    "DIVISION",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "DayOfYear",
    "Lag_3",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

# Define the target variable
target_column = "Intervention_Count"

# Verify the selected columns
print("Number of predictive features:", len(feature_columns))
print("Target variable:", target_column)

print("\nPredictive features:")
for column in feature_columns:
    print("-", column)


Number of predictive features: 12
Target variable: Intervention_Count

Predictive features:
- DIVISION
- DayOfWeek
- Month
- DayOfMonth
- IsWeekend
- DayOfYear
- Lag_3
- Lag_7
- Lag_14
- Lag_28
- Rolling_Mean_7
- Rolling_Mean_14


## 5.5. Prepare the First Training and Validation Window

We will use January 2021 as the first example to understand the model training process.

The training dataset will include observations through December 29, 2020, following the D−3 cutoff. The validation dataset will contain observations from January 1 through January 31, 2021.

We will separate predictive features (X) from known intervention counts (y).

The validation target values will be used only to evaluate the predictions, not to train the model.

In [ ]:

# Select the first validation window
first_window = df_schedule.iloc[0]

# Define the training and validation periods
training_cutoff = first_window["Training_Cutoff"]
validation_start = first_window["Validation_Start"]
validation_end = first_window["Validation_End"]

# Select training observations
df_train = df_modeling[
    df_modeling["Date"] <= training_cutoff
].copy()

# Select validation observations
df_val = df_modeling[
    (df_modeling["Date"] >= validation_start) &
    (df_modeling["Date"] <= validation_end)
].copy()

# Separate features and target
X_train = df_train[feature_columns]
y_train = df_train[target_column]

X_val = df_val[feature_columns]
y_val = df_val[target_column]

# Verify the results
print("Training cutoff:", training_cutoff.date())
print("Validation period:", validation_start.date(), "to", validation_end.date())

print("\nX_train dimensions:", X_train.shape)
print("y_train observations:", len(y_train))

print("X_val dimensions:", X_val.shape)
print("y_val observations:", len(y_val))

print("\nMissing training targets:", y_train.isna().sum())
print("Missing validation targets:", y_val.isna().sum())


Training cutoff: 2020-12-29
Validation period: 2021-01-01 to 2021-01-31

X_train dimensions: (2184, 12)
y_train observations: 2184
X_val dimensions: (186, 12)
y_val observations: 186

Missing training targets: 0
Missing validation targets: 0


In [ ]:

# Verify chronological separation
assert df_train["Date"].max() <= training_cutoff
assert df_val["Date"].min() == validation_start
assert df_val["Date"].max() == validation_end

# Verify that training and validation dates do not overlap
assert df_train["Date"].max() < df_val["Date"].min()

# Verify that both datasets contain all required features
assert all(col in df_train.columns for col in feature_columns)
assert all(col in df_val.columns for col in feature_columns)

# Check missing feature values
print("Missing training features:")
print(X_train.isna().sum())

print("\nMissing validation features:")
print(X_val.isna().sum())

print("\nChronological separation: PASSED")


Missing training features:
DIVISION             0
DayOfWeek            0
Month                0
DayOfMonth           0
IsWeekend            0
DayOfYear            0
Lag_3               18
Lag_7               42
Lag_14              84
Lag_28             168
Rolling_Mean_7      54
Rolling_Mean_14     96
dtype: int64

Missing validation features:
DIVISION           0
DayOfWeek          0
Month              0
DayOfMonth         0
IsWeekend          0
DayOfYear          0
Lag_3              0
Lag_7              0
Lag_14             0
Lag_28             0
Rolling_Mean_7     0
Rolling_Mean_14    0
dtype: int64

Chronological separation: PASSED


**Methodological Observation**

The first validation window contains 2,184 training observations and 186 validation observations.

The chronological separation checks passed, confirming that the training and validation periods do not overlap.

Missing training feature values appear only in historical lag and rolling average features. Their counts match the expected unavailable history at the beginning of the dataset.

All 186 validation observations have complete predictive features.

The historical feature definitions from Stage 3 are consistent with the D−3 information cutoff. The current checks support the validity of the first training and validation split, although they do not independently verify every exported historical feature value.

We will preserve the missing training features because LightGBM can handle them natively. Other models may require different preprocessing.

## 5.6. Train the First LightGBM Model

We will train an initial LightGBM regression model using historical observations available through December 29, 2020.

The model will learn relationships between the 12 candidate predictive features and daily intervention counts.

We will use a simple initial configuration without hyperparameter tuning. This first model is intended to demonstrate the training and prediction process before applying the monthly validation framework.

The model will be evaluated on January 2021 observations that were not used during training.

In [ ]:

# Prepare categorical features for LightGBM
X_train = X_train.copy()
X_val = X_val.copy()

# Treat DIVISION as a categorical feature
X_train["DIVISION"] = X_train["DIVISION"].astype("category")
X_val["DIVISION"] = X_val["DIVISION"].astype("category")

# Verify the data type
print("DIVISION data type:", X_train["DIVISION"].dtype)


DIVISION data type: category


In [ ]:


# Create the LightGBM regression model
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1
)

# Train the model
model.fit(
    X_train,
    y_train,
    categorical_feature=["DIVISION"]
)

# Generate predictions
y_pred = model.predict(X_val)

# Display the first predictions
print("Model training completed.")
print("Number of predictions:", len(y_pred))

df_predictions = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_predictions["Predicted_Count"] = y_pred

display(df_predictions.head(10))



Model training completed.
Number of predictions: 186


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,23.222282
2197,2021-01-01,2,41.0,53.352678
2198,2021-01-01,3,48.0,55.816048
2199,2021-01-01,4,53.0,59.123836
2200,2021-01-01,5,42.0,57.046466
2201,2021-01-01,6,27.0,21.340081
2202,2021-01-02,1,22.0,22.547950
2203,2021-01-02,2,35.0,52.847560
2204,2021-01-02,3,48.0,59.675842
2205,2021-01-02,4,48.0,59.882612


**Methodological Observation**

The DIVISION feature identifies six operational categories rather than numerical quantities.

We updated the initial LightGBM implementation to treat DIVISION as a categorical feature using LightGBM's native categorical feature support.

The updated model completed training and generated 186 predictions for January 2021.

The predictions differ from those produced when DIVISION was treated as numerical. However, this initial comparison does not establish which representation provides better overall forecasting accuracy.

The categorical representation will be applied consistently during monthly validation. Model performance will be evaluated using MAE and Bias.

## 5.7. Evaluate the First LightGBM Model

We will evaluate the initial LightGBM model on January 2021 observations.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

These results will provide an initial example of model performance. They will not be used as the final validation result or as evidence that LightGBM outperforms the baseline across the full development period.

In [ ]:

# Calculate prediction errors
df_predictions["Error"] = (
    df_predictions["Predicted_Count"] -
    df_predictions["Intervention_Count"]
)

# Calculate evaluation metrics
mae = mean_absolute_error(
    df_predictions["Intervention_Count"],
    df_predictions["Predicted_Count"]
)

bias = df_predictions["Error"].mean()

# Display the results
print("Validation period: January 2021")
print("LightGBM MAE:", round(mae, 3))
print("LightGBM Bias:", round(bias, 3))

# Calculate MAE for each operational division
division_mae = (
    df_predictions.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(division_mae)


Validation period: January 2021
LightGBM MAE: 7.781
LightGBM Bias: 3.127

MAE by division:


DIVISION
1     5.354374
2     7.387283
3     7.549803
4     9.363309
5    12.341064
6     4.688350
Name: Error, dtype: float64

**Methodological Observation**

The initial LightGBM model was trained using 2,184 historical observations and evaluated on 186 observations from January 2021.

The model treats DIVISION as a categorical feature and uses the 12 candidate predictive features developed in Stage 3.

The model achieved an MAE of 7.781 interventions and a positive Bias of 3.127 interventions, indicating a tendency to overestimate daily intervention counts.

The lowest division MAE was observed in Division 6 (4.688), while the highest was observed in Division 5 (12.341).

Compared with the initial numerical representation of DIVISION, the categorical representation produced approximately the same overall MAE but a higher positive Bias. Performance differences varied across divisions.

These results do not establish that either representation provides better overall forecasting performance. The categorical representation will be retained because it reflects the meaning of DIVISION as an operational category.

The model will be evaluated across the full chronological validation period before drawing conclusions about its overall performance or comparing it with the baseline.

## 5.8. Train LightGBM Across Monthly Validation Windows

We will apply the initial LightGBM configuration to all 48 monthly validation windows from January 2021 through December 2024.

For each month, a new model will be trained using historical observations available through the corresponding D−3 training cutoff. The model will remain fixed throughout the validation month.

**The DIVISION feature will be treated as categorical in every monthly LightGBM model.**

Daily validation features may incorporate newly available historical intervention counts, provided that they respect the D−3 information cutoff.

We will store all out-of-sample predictions for subsequent evaluation against the selected baseline. The reserved final test period will remain excluded.

In [ ]:


# Store predictions from all validation months
all_predictions = []

# Define the six operational divisions as categories
division_categories = [1, 2, 3, 4, 5, 6]

# Repeat training and prediction for each month
for _, window in df_schedule.iterrows():

    # Select training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Prepare features and target
    X_train_month = monthly_train[feature_columns].copy()
    y_train_month = monthly_train[target_column]

    X_val_month = monthly_val[feature_columns].copy()

    # Treat DIVISION as a categorical feature
    X_train_month["DIVISION"] = pd.Categorical(
        X_train_month["DIVISION"],
        categories=division_categories
    )

    X_val_month["DIVISION"] = pd.Categorical(
        X_val_month["DIVISION"],
        categories=division_categories
    )

    # Create a new model for this month
    monthly_model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=15,
        random_state=42,
        verbosity=-1
    )

    # Train the monthly model
    monthly_model.fit(
        X_train_month,
        y_train_month,
        categorical_feature=["DIVISION"]
    )

    # Generate validation predictions
    monthly_pred = monthly_model.predict(X_val_month)

    # Store predictions and actual counts
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Predicted_Count"] = monthly_pred

    all_predictions.append(monthly_results)

# Combine monthly predictions
df_lightgbm_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

# Verify results
print("Validation months processed:", len(all_predictions))
print("Total predictions:", len(df_lightgbm_predictions))
print(
    "Missing predictions:",
    df_lightgbm_predictions["Predicted_Count"].isna().sum()
)

print("\nFirst five predictions:")
display(df_lightgbm_predictions.head())



Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Predicted_Count
0,2021-01-01,1,18.0,23.222282
1,2021-01-01,2,41.0,53.352678
2,2021-01-01,3,48.0,55.816048
3,2021-01-01,4,53.0,59.123836
4,2021-01-01,5,42.0,57.046466


In [ ]:

# Verify the monthly validation results
assert len(all_predictions) == len(df_schedule)
assert len(df_lightgbm_predictions) == 8760
assert df_lightgbm_predictions["Predicted_Count"].notna().all()

# Check that each division and date appears only once
assert not df_lightgbm_predictions.duplicated(
    subset=["Date", "DIVISION"]
).any()

# Confirm the validation period
assert df_lightgbm_predictions["Date"].min() == pd.Timestamp("2021-01-01")
assert df_lightgbm_predictions["Date"].max() == pd.Timestamp("2024-12-30")

# Confirm that every prediction has a known target
assert df_lightgbm_predictions["Intervention_Count"].notna().all()

print("Monthly validation checks: PASSED")


Monthly validation checks: PASSED


**Methodological Observation**

LightGBM was trained and evaluated using 48 monthly expanding validation windows from January 2021 through December 2024.

For each month, a new model was trained using historical observations available through the corresponding training cutoff. Each fitted model remained unchanged throughout its validation month.

DIVISION was treated as a categorical feature in all monthly models, using the same six operational categories.

The process generated 8,760 predictions, with no missing prediction values. The additional validation checks passed, confirming the expected date range, unique division and date combinations, and complete target values.

The first predictions matched those obtained during the initial January 2021 training example.

The resulting predictions will be used to evaluate LightGBM performance across the development period. The 2025–2026 final test period remains reserved.

## 5.9. Evaluate LightGBM Validation Performance

We will evaluate the LightGBM predictions generated across the 48 monthly validation windows from 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also examine MAE and Bias by operational division and calendar year to identify differences in forecasting performance.

These results will be compared with the selected baseline using the same validation observations. The reserved final test period will remain excluded.

In [ ]:

# Calculate prediction errors
df_lightgbm_predictions["Error"] = (
    df_lightgbm_predictions["Predicted_Count"] -
    df_lightgbm_predictions["Intervention_Count"]
)

# Calculate the overall validation metrics
overall_mae = df_lightgbm_predictions["Error"].abs().mean()
overall_bias = df_lightgbm_predictions["Error"].mean()

print("LightGBM Validation Results (2021-2024)")
print("Total predictions:", len(df_lightgbm_predictions))
print("Overall MAE:", round(overall_mae, 3))
print("Overall Bias:", round(overall_bias, 3))

# Calculate performance by division
division_metrics = (
    df_lightgbm_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(division_metrics.round(3))

# Extract the year from each forecast date
df_lightgbm_predictions["Year"] = (
    df_lightgbm_predictions["Date"].dt.year
)

# Calculate performance by year
yearly_metrics = (
    df_lightgbm_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(yearly_metrics.round(3))


LightGBM Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.147
Overall Bias: -2.002

Performance by division:


,MAE,Bias
DIVISION,,
1,6.032,-1.373
2,8.242,-1.709
3,9.419,-3.313
4,8.763,-2.114
5,8.866,-2.050
6,7.558,-1.452



Performance by year:


,MAE,Bias
Year,,
2021,7.187,-0.934
2022,7.702,-2.419
2023,9.755,-3.645
2024,7.943,-1.009


**Methodological Observation**

LightGBM was evaluated using 8,760 predictions generated across 48 monthly validation windows from January 2021 through December 2024.

The model treated DIVISION as a categorical feature in all monthly training windows.

The overall MAE was 8.147 interventions, and the overall Bias was −2.002 interventions, indicating a tendency to underestimate daily intervention counts.

Division 1 had the lowest MAE (6.032), while Division 3 had the highest MAE (9.419) and the strongest negative Bias (−3.313).

All six divisions showed negative prediction Bias.

Annual performance varied across the validation period. The highest MAE was observed in 2023 (9.755), which also had the strongest negative Bias (−3.645).

Compared with treating DIVISION as numerical, the categorical representation produced a slightly lower overall MAE but a slightly more negative Bias. The differences were small.

These findings describe LightGBM's chronological validation performance. A comparison with the selected baseline and the other forecasting methods is required before making a model selection.

## 5.10. Install and Import CatBoost

We will introduce CatBoost as the second machine learning model for forecasting daily SIM intervention counts.

Like LightGBM, CatBoost uses gradient-boosted decision trees. One important difference is its ability to handle categorical predictors directly.

We will treat `DIVISION` as a categorical feature because operational division identifiers represent distinct categories rather than numerical quantities.

CatBoost will be evaluated using the same chronological validation schedule and target variable as LightGBM.

In [ ]:

# Install CatBoost
%pip install catboost


/home/pliqui/.cache/uv/builds-v0/.tmpz6VwEt/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:

# Import CatBoost
from catboost import CatBoostRegressor

# Verify the installation
import catboost

print("CatBoost version:", catboost.__version__)


CatBoost version: 1.2.10


## 5.11. Prepare Categorical Features for CatBoost

We will prepare the predictive features for the initial CatBoost model.

The `DIVISION` column represents six operational categories rather than a continuous numerical quantity. We will convert this column to a string and identify it as a categorical feature for CatBoost.

The remaining predictive features will retain their numerical values, including any missing historical values.

We will use the same January 2021 training and validation observations previously used for the initial LightGBM model to maintain a consistent evaluation setup.

In [ ]:

# Create separate feature datasets for CatBoost
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()

# Convert DIVISION into a categorical identifier
X_train_cat["DIVISION"] = X_train_cat["DIVISION"].astype(str)
X_val_cat["DIVISION"] = X_val_cat["DIVISION"].astype(str)

# Identify the categorical feature
categorical_features = ["DIVISION"]

# Verify the categorical values
print("Categorical features:", categorical_features)

print(
    "Training divisions:",
    sorted(X_train_cat["DIVISION"].unique())
)

print(
    "Validation divisions:",
    sorted(X_val_cat["DIVISION"].unique())
)

# Verify dataset dimensions
print("\nTraining features:", X_train_cat.shape)
print("Validation features:", X_val_cat.shape)

# Verify missing numerical values
print(
    "Missing training feature values:",
    X_train_cat.isna().sum().sum()
)

print(
    "Missing validation feature values:",
    X_val_cat.isna().sum().sum()
)


Categorical features: ['DIVISION']
Training divisions: ['1', '2', '3', '4', '5', '6']
Validation divisions: ['1', '2', '3', '4', '5', '6']

Training features: (2184, 12)
Validation features: (186, 12)
Missing training feature values: 462
Missing validation feature values: 0


## 5.12. Train the First CatBoost Model

We will train an initial CatBoost regression model using historical observations available through December 29, 2020.

The model will use the same 12 candidate predictors and target variable as LightGBM, with `DIVISION` explicitly treated as categorical.

We will begin with a simple configuration without hyperparameter tuning.

**CatBoost will use Root Mean Squared Error (RMSE) as its training loss function. RMSE gives greater weight to large prediction errors.**

**We will evaluate the model using MAE and Bias, following the evaluation metrics selected for this project. The training loss and evaluation metrics serve different purposes and do not need to be identical.**

The trained model will generate predictions for January 2021. These predictions will allow us to evaluate its initial performance before applying the complete monthly validation schedule.

In [ ]:

# Create the CatBoost regression model
catboost_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.05,
    depth=4,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    thread_count=2
)

# Train the model
catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features
)

# Generate predictions for January 2021
catboost_predictions = catboost_model.predict(X_val_cat)

# Prepare the prediction results
df_catboost_first = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_catboost_first["Predicted_Count"] = catboost_predictions

# Display results
print("CatBoost training completed.")
print("Number of predictions:", len(catboost_predictions))

display(df_catboost_first.head(10))


CatBoost training completed.
Number of predictions: 186


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,25.721262
2197,2021-01-01,2,41.0,54.572590
2198,2021-01-01,3,48.0,53.516553
2199,2021-01-01,4,53.0,56.256852
2200,2021-01-01,5,42.0,56.173112
2201,2021-01-01,6,27.0,28.867867
2202,2021-01-02,1,22.0,27.957362
2203,2021-01-02,2,35.0,52.738028
2204,2021-01-02,3,48.0,54.030654
2205,2021-01-02,4,48.0,56.653569


## 5.13. Evaluate the First CatBoost Model

We will evaluate the initial CatBoost model using the 186 validation observations from January 2021.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also calculate MAE separately for each operational division.

These results provide an initial assessment of CatBoost performance. The complete 2021–2024 monthly validation will be required before comparing the models across the full development period.

In [ ]:

# Calculate prediction errors
df_catboost_first["Error"] = (
    df_catboost_first["Predicted_Count"] -
    df_catboost_first["Intervention_Count"]
)

# Calculate overall MAE
catboost_mae = mean_absolute_error(
    df_catboost_first["Intervention_Count"],
    df_catboost_first["Predicted_Count"]
)

# Calculate overall prediction bias
catboost_bias = df_catboost_first["Error"].mean()

# Display the results
print("CatBoost Validation Results (January 2021)")
print("Total predictions:", len(df_catboost_first))
print("MAE:", round(catboost_mae, 3))
print("Bias:", round(catboost_bias, 3))

# Calculate MAE for each division
catboost_division_mae = (
    df_catboost_first.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(catboost_division_mae.round(3))


CatBoost Validation Results (January 2021)
Total predictions: 186
MAE: 7.556
Bias: 3.396

MAE by division:


DIVISION
1     6.872
2     8.040
3     5.878
4     8.197
5    12.015
6     4.334
Name: Error, dtype: float64

**Methodological Observation**

The initial CatBoost model generated 186 predictions for January 2021, covering all six operational divisions.

The model achieved an MAE of 7.556 interventions and a positive Bias of 3.396 interventions, indicating an average tendency to overestimate daily intervention counts.

Division 6 had the lowest MAE (4.334), while Division 5 had the highest MAE (12.015).

These results provide an initial evaluation of CatBoost performance. The complete monthly validation from 2021 through 2024 is required to assess its performance over a longer period.


## 5.14. Train CatBoost Across Monthly Validation Windows

We will train CatBoost models using the 48 monthly chronological validation windows from January 2021 through December 2024.

For each month, a new model will be trained using observations available through the corresponding D−3 training cutoff.

The model will remain fixed within each validation month. Daily predictive features may incorporate newly available historical intervention counts, respecting the D−3 information restriction.

The same 12 candidate predictive features and initial CatBoost configuration will be used throughout the validation period. `DIVISION` will be treated as a categorical predictor.

All validation predictions will be stored for model comparison in Stage 6.

In [ ]:

# Store predictions from all validation months
all_catboost_predictions = []

# Train and evaluate a new model for each month
for _, window in df_schedule.iterrows():

    # Select training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Prepare the training and validation features
    X_train_month = monthly_train[feature_columns].copy()
    X_val_month = monthly_val[feature_columns].copy()

    # Convert DIVISION to a categorical identifier
    X_train_month["DIVISION"] = X_train_month["DIVISION"].astype(str)
    X_val_month["DIVISION"] = X_val_month["DIVISION"].astype(str)

    # Prepare the training target
    y_train_month = monthly_train[target_column]

    # Create a new CatBoost model
    monthly_catboost_model = CatBoostRegressor(
        iterations=100,
        learning_rate=0.05,
        depth=4,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=2
    )

    # Train the model
    monthly_catboost_model.fit(
        X_train_month,
        y_train_month,
        cat_features=categorical_features
    )

    # Generate predictions for the validation month
    monthly_predictions = monthly_catboost_model.predict(X_val_month)

    # Store forecast dates, divisions, actuals, and predictions
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Predicted_Count"] = monthly_predictions

    all_catboost_predictions.append(monthly_results)

# Combine predictions from all validation months
df_catboost_predictions = pd.concat(
    all_catboost_predictions,
    ignore_index=True
)

# Verify the results
print("Validation months processed:", len(all_catboost_predictions))
print("Total predictions:", len(df_catboost_predictions))
print(
    "Missing predictions:",
    df_catboost_predictions["Predicted_Count"].isna().sum()
)

print("\nFirst five predictions:")
display(df_catboost_predictions.head())


Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Predicted_Count
0,2021-01-01,1,18.0,25.721262
1,2021-01-01,2,41.0,54.572590
2,2021-01-01,3,48.0,53.516553
3,2021-01-01,4,53.0,56.256852
4,2021-01-01,5,42.0,56.173112


In [ ]:

# Verify the monthly CatBoost validation results
assert len(all_catboost_predictions) == len(df_schedule)
assert len(df_catboost_predictions) == 8760
assert df_catboost_predictions["Predicted_Count"].notna().all()

# Check that each division and date appears only once
assert not df_catboost_predictions.duplicated(
    subset=["Date", "DIVISION"]
).any()

# Confirm the validation period
assert df_catboost_predictions["Date"].min() == pd.Timestamp("2021-01-01")
assert df_catboost_predictions["Date"].max() == pd.Timestamp("2024-12-30")

# Confirm that every prediction has a known target
assert df_catboost_predictions["Intervention_Count"].notna().all()

print("CatBoost monthly validation checks: PASSED")


CatBoost monthly validation checks: PASSED


**Methodological Observation**

CatBoost was trained using 48 monthly expanding validation windows from January 2021 through December 2024.

For each month, a new model was trained using historical observations available through the corresponding training cutoff. Each fitted model remained unchanged during its validation month.

The same 12 predictive features and initial CatBoost configuration were used throughout the validation period. DIVISION was treated as a categorical feature in every monthly model.

The process generated 8,760 predictions with no missing prediction values.

All additional validation checks passed, confirming the expected date range, unique division and date combinations, and complete target values.

The first predictions matched those obtained during the initial January 2021 training example.

The predictions will be used to evaluate CatBoost performance across the complete validation period. The 2025–2026 final test period remains reserved.

## 5.15. Evaluate CatBoost Validation Performance

We will evaluate CatBoost predictions across the 48 monthly validation windows covering 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether CatBoost tends to overestimate or underestimate daily intervention counts.

We will also examine performance separately by operational division and calendar year to identify differences in forecasting accuracy.

These results describe CatBoost performance individually. The reserved final test period will remain excluded.

In [ ]:

# Calculate prediction errors
df_catboost_predictions["Error"] = (
    df_catboost_predictions["Predicted_Count"] -
    df_catboost_predictions["Intervention_Count"]
)

# Calculate overall validation metrics
catboost_overall_mae = (
    df_catboost_predictions["Error"].abs().mean()
)

catboost_overall_bias = (
    df_catboost_predictions["Error"].mean()
)

# Display overall results
print("CatBoost Validation Results (2021-2024)")
print("Total predictions:", len(df_catboost_predictions))
print("Overall MAE:", round(catboost_overall_mae, 3))
print("Overall Bias:", round(catboost_overall_bias, 3))

# Calculate performance by division
catboost_division_metrics = (
    df_catboost_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(catboost_division_metrics.round(3))

# Extract the year from each forecast date
df_catboost_predictions["Year"] = (
    df_catboost_predictions["Date"].dt.year
)

# Calculate performance by year
catboost_yearly_metrics = (
    df_catboost_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(catboost_yearly_metrics.round(3))


CatBoost Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.127
Overall Bias: -1.768

Performance by division:


,MAE,Bias
DIVISION,,
1,6.134,-0.892
2,8.143,-0.867
3,9.322,-3.806
4,8.506,-2.565
5,8.789,-1.946
6,7.869,-0.530



Performance by year:


,MAE,Bias
Year,,
2021,7.123,-0.846
2022,7.550,-1.945
2023,9.958,-3.184
2024,7.878,-1.096


**Methodological Observation**

CatBoost was evaluated using 8,760 predictions generated across 48 monthly validation windows from January 2021 through December 2024.

The overall MAE was 8.127 interventions, and the overall Bias was −1.768 interventions, indicating an average tendency to underestimate daily intervention counts.

Division 1 had the lowest MAE (6.134), while Division 3 had the highest MAE (9.322) and the strongest negative Bias (−3.806).

All six operational divisions showed negative prediction Bias.

Annual performance varied during the validation period. The lowest MAE was observed in 2021 (7.123), while the highest was observed in 2023 (9.958). The year 2023 also showed the strongest negative annual Bias (−3.184).

These findings show that CatBoost forecasting performance was not uniform across divisions or years. The results do not establish the causes of these differences.

The model was retrained monthly using expanding historical data, but prediction errors still varied over time.

These results describe CatBoost's individual chronological validation performance. Model comparison and selection will be performed in Stage 6.

## 5.16. Introduce Linear Regression

We will train Linear Regression as a simpler, more interpretable model using the same features and chronological validation windows as LightGBM and CatBoost. Its predictions will be saved for model comparison in Stage 6.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


## 5.17. Define Features and Preprocessing Pipeline

We will prepare the 12 candidate predictive features for Linear Regression.

The `DIVISION` variable will be encoded using One-Hot Encoding because division identifiers represent categories rather than continuous numerical values.

Missing numerical feature values will be replaced using the median calculated from the training observations.

A scikit-learn preprocessing pipeline will ensure that these transformations are fitted only on the training data for each chronological validation window.

This approach will allow Linear Regression to use the same candidate predictors as LightGBM and CatBoost while preventing preprocessing-related data leakage.

In [ ]:

# Define categorical features
categorical_features = ["DIVISION"]

# Define numerical features
numerical_features = [
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "DayOfYear",
    "Lag_3",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

# Combine all predictive features
feature_columns = categorical_features + numerical_features

# Create numerical preprocessing
numerical_preprocessing = SimpleImputer(
    strategy="median"
)

# Create categorical preprocessing
categorical_preprocessing = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing operations
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preprocessing, numerical_features),
        ("categorical", categorical_preprocessing, categorical_features)
    ]
)

# Create the Linear Regression pipeline
linear_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("regression", LinearRegression())
    ]
)

# Verify feature configuration
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))
print("Total predictive features:", len(feature_columns))

print("\nModel pipeline:")
print(linear_model)


Categorical features: 1
Numerical features: 11
Total predictive features: 12

Model pipeline:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  SimpleImputer(strategy='median'),
                                                  ['DayOfWeek', 'Month',
                                                   'DayOfMonth', 'IsWeekend',
                                                   'DayOfYear', 'Lag_3',
                                                   'Lag_7', 'Lag_14', 'Lag_28',
                                                   'Rolling_Mean_7',
                                                   'Rolling_Mean_14']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['DIVISION'])])),
                ('regression', LinearRegression())])


## 5.18. Train the First Linear Regression Model

We will train an initial Linear Regression model using historical observations available through December 29, 2020.

The model will be evaluated on January 2021, using the same chronological training and validation periods as the initial LightGBM and CatBoost models.

All preprocessing operations will be fitted using training observations only.

We will first verify that the model generates predictions successfully before calculating its forecasting accuracy.

In [ ]:

# Select the first validation window
first_window = df_schedule.iloc[0]

# Define the training and validation periods
training_cutoff = first_window["Training_Cutoff"]
validation_start = first_window["Validation_Start"]
validation_end = first_window["Validation_End"]

# Select training observations
df_train = df_modeling[
    df_modeling["Date"] <= training_cutoff
].copy()

# Select validation observations
df_val = df_modeling[
    (df_modeling["Date"] >= validation_start) &
    (df_modeling["Date"] <= validation_end)
].copy()

# Prepare input features and target
X_train = df_train[feature_columns]
y_train = df_train["Intervention_Count"]

X_val = df_val[feature_columns]

# Train Linear Regression
linear_model.fit(X_train, y_train)

# Generate predictions
linear_predictions = linear_model.predict(X_val)

# Store the results
df_linear_first = df_val[
    ["Date", "DIVISION", "Intervention_Count"]
].copy()

df_linear_first["Predicted_Count"] = linear_predictions

# Display results
print("Linear Regression training completed.")
print("Training observations:", len(df_train))
print("Validation observations:", len(df_val))
print("Number of predictions:", len(linear_predictions))
print("Missing predictions:", np.isnan(linear_predictions).sum())

display(df_linear_first.head(10))


Linear Regression training completed.
Training observations: 2184
Validation observations: 186
Number of predictions: 186
Missing predictions: 0


,Date,DIVISION,Intervention_Count,Predicted_Count
2196,2021-01-01,1,18.0,26.262695
2197,2021-01-01,2,41.0,52.819793
2198,2021-01-01,3,48.0,51.549891
2199,2021-01-01,4,53.0,56.753947
2200,2021-01-01,5,42.0,55.034881
2201,2021-01-01,6,27.0,27.083542
2202,2021-01-02,1,22.0,26.406278
2203,2021-01-02,2,35.0,49.296900
2204,2021-01-02,3,48.0,49.323978
2205,2021-01-02,4,48.0,58.307746


## 5.19. Evaluate the First Linear Regression Model

We will evaluate the initial Linear Regression model using the 186 validation observations from January 2021.

Mean Absolute Error (MAE) will measure the average magnitude of prediction errors. Prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also calculate MAE by operational division to examine differences in forecasting performance.

These initial results demonstrate the first Linear Regression validation month. The complete 48-month validation will be used to assess its overall performance.

In [ ]:

# Calculate prediction errors
df_linear_first["Error"] = (
    df_linear_first["Predicted_Count"] -
    df_linear_first["Intervention_Count"]
)

# Calculate overall MAE and Bias
linear_mae = df_linear_first["Error"].abs().mean()
linear_bias = df_linear_first["Error"].mean()

print("Linear Regression Validation Results (January 2021)")
print("Total predictions:", len(df_linear_first))
print("MAE:", round(linear_mae, 3))
print("Bias:", round(linear_bias, 3))

# Calculate MAE by division
linear_division_mae = (
    df_linear_first.groupby("DIVISION")["Error"]
    .apply(lambda errors: errors.abs().mean())
)

print("\nMAE by division:")
display(linear_division_mae.round(3))


Linear Regression Validation Results (January 2021)
Total predictions: 186
MAE: 6.918
Bias: 0.639

MAE by division:


DIVISION
1     5.494
2     7.149
3     5.610
4     7.671
5    11.429
6     4.157
Name: Error, dtype: float64

**Methodological Observation**

The initial Linear Regression model was evaluated using 186 predictions for January 2021, covering all six operational divisions.

The model achieved an MAE of 6.918 interventions and a positive Bias of 0.639 interventions, indicating a small average tendency to overestimate daily intervention counts.

Division 6 had the lowest MAE (4.157), while Division 5 had the highest MAE (11.429).

These results show differences in forecasting accuracy across the six divisions during the initial validation month.

The overall Bias was relatively close to zero, but this does not guarantee that prediction errors were consistently small or that individual divisions had no systematic bias.

The complete monthly validation from 2021 through 2024 is required to evaluate Linear Regression performance over a longer period.

## 5.20. Train Linear Regression Across Monthly Validation Windows

We will evaluate Linear Regression using the same 48 monthly chronological validation windows applied to LightGBM and CatBoost.

For each validation month, a new Linear Regression pipeline will be fitted using historical observations available through the corresponding D−3 training cutoff.

The preprocessing pipeline will calculate missing-value replacements and categorical encodings using training data only.

The model will remain fixed during each validation month, while daily predictive features may use newly available historical intervention counts through D−3.

All validation predictions will be stored for subsequent comparison with the baseline and previously evaluated machine learning models. The reserved 2025–2026 final test period will remain excluded.

In [ ]:

from sklearn.base import clone

# Store predictions from all validation months
all_linear_predictions = []

# Process each monthly validation window
for _, window in df_schedule.iterrows():

    # Select historical training observations
    monthly_train = df_modeling[
        df_modeling["Date"] <= window["Training_Cutoff"]
    ]

    # Select validation observations
    monthly_val = df_modeling[
        (df_modeling["Date"] >= window["Validation_Start"]) &
        (df_modeling["Date"] <= window["Validation_End"])
    ]

    # Separate features and target
    X_train_month = monthly_train[feature_columns]
    y_train_month = monthly_train["Intervention_Count"]

    X_val_month = monthly_val[feature_columns]

    # Create a fresh copy of the full pipeline
    monthly_linear_model = clone(linear_model)

    # Fit preprocessing and regression using training data only
    monthly_linear_model.fit(X_train_month, y_train_month)

    # Generate predictions for the validation month
    monthly_predictions = monthly_linear_model.predict(X_val_month)

    # Store the predictions and actual counts
    monthly_results = monthly_val[
        ["Date", "DIVISION", "Intervention_Count"]
    ].copy()

    monthly_results["Linear_Prediction"] = monthly_predictions

    all_linear_predictions.append(monthly_results)

# Combine all monthly predictions
df_linear_predictions = pd.concat(
    all_linear_predictions,
    ignore_index=True
)

# Verify results
print("Validation months processed:", len(all_linear_predictions))
print("Total predictions:", len(df_linear_predictions))

print(
    "Missing predictions:",
    df_linear_predictions["Linear_Prediction"].isna().sum()
)

print("\nFirst five predictions:")
display(df_linear_predictions.head())


Validation months processed: 48
Total predictions: 8760
Missing predictions: 0

First five predictions:


,Date,DIVISION,Intervention_Count,Linear_Prediction
0,2021-01-01,1,18.0,26.262695
1,2021-01-01,2,41.0,52.819793
2,2021-01-01,3,48.0,51.549891
3,2021-01-01,4,53.0,56.753947
4,2021-01-01,5,42.0,55.034881


In [ ]:

# Verify monthly Linear Regression results
assert len(all_linear_predictions) == len(df_schedule)
assert len(df_linear_predictions) == 8760

# Check missing values
assert df_linear_predictions["Linear_Prediction"].notna().all()
assert df_linear_predictions["Intervention_Count"].notna().all()

# Check unique observations
assert not df_linear_predictions.duplicated(
    subset=["Date", "DIVISION"]
).any()

# Check the validation date range
assert df_linear_predictions["Date"].min() == pd.Timestamp("2021-01-01")
assert df_linear_predictions["Date"].max() == pd.Timestamp("2024-12-30")

# Check consistency with the initial January 2021 model
first_month_predictions = df_linear_predictions[
    (df_linear_predictions["Date"] >= "2021-01-01") &
    (df_linear_predictions["Date"] <= "2021-01-31")
]

assert np.allclose(
    first_month_predictions["Linear_Prediction"].to_numpy(),
    df_linear_first["Predicted_Count"].to_numpy()
)

print("Linear Regression monthly validation checks: PASSED")


Linear Regression monthly validation checks: PASSED


**Methodological Observation**

Linear Regression was trained using 48 monthly expanding validation windows from January 2021 through December 2024.

For each month, a new preprocessing and regression pipeline was fitted using historical observations available through the corresponding training cutoff. The fitted pipeline remained unchanged during the validation month.

Missing numerical values were replaced using medians calculated from the training data, and DIVISION was encoded using One-Hot Encoding. These preprocessing steps were fitted separately for each monthly training window.

The process generated 8,760 predictions with no missing prediction values.

All additional validation checks passed, confirming the expected date range, unique division and date combinations, complete target values, and consistency with the initial January 2021 predictions.

These results confirm the successful execution of the monthly validation procedure. Forecasting accuracy will be evaluated in the next section.

The 2025–2026 final test period remains reserved.

## 5.21. Evaluate Linear Regression Validation Performance

We will evaluate Linear Regression predictions across the 48 monthly chronological validation windows covering 2021 through 2024.

Mean Absolute Error (MAE) will measure the average magnitude of forecasting errors, while prediction bias will indicate whether the model tends to overestimate or underestimate intervention counts.

We will also evaluate performance separately by operational division and calendar year to examine differences in forecasting accuracy and temporal stability.

These results will provide the evidence needed to assess whether Linear Regression is a competitive alternative to LightGBM, CatBoost, and the historical baseline.

In [ ]:

# Calculate prediction errors
df_linear_predictions["Error"] = (
    df_linear_predictions["Linear_Prediction"] -
    df_linear_predictions["Intervention_Count"]
)

# Calculate overall MAE and Bias
linear_overall_mae = (
    df_linear_predictions["Error"].abs().mean()
)

linear_overall_bias = (
    df_linear_predictions["Error"].mean()
)

# Display overall results
print("Linear Regression Validation Results (2021-2024)")
print("Total predictions:", len(df_linear_predictions))
print("Overall MAE:", round(linear_overall_mae, 3))
print("Overall Bias:", round(linear_overall_bias, 3))

# Calculate performance by division
linear_division_metrics = (
    df_linear_predictions.groupby("DIVISION")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by division:")
display(linear_division_metrics.round(3))

# Extract the forecast year
df_linear_predictions["Year"] = (
    df_linear_predictions["Date"].dt.year
)

# Calculate performance by year
linear_yearly_metrics = (
    df_linear_predictions.groupby("Year")["Error"]
    .agg(
        MAE=lambda errors: errors.abs().mean(),
        Bias="mean"
    )
)

print("\nPerformance by year:")
display(linear_yearly_metrics.round(3))


Linear Regression Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.384
Overall Bias: -1.34

Performance by division:


,MAE,Bias
DIVISION,,
1,6.413,-1.079
2,8.443,-1.251
3,9.716,-1.521
4,8.780,-1.200
5,9.150,-1.255
6,7.805,-1.732



Performance by year:


,MAE,Bias
Year,,
2021,7.292,-0.928
2022,7.568,-0.965
2023,10.619,-1.520
2024,8.059,-1.945


**Methodological Observation**

Linear Regression was evaluated using 8,760 predictions generated across 48 monthly chronological validation windows from January 2021 through December 2024.

The overall MAE was 8.384 interventions, and the overall Bias was −1.340 interventions, indicating an average tendency to underestimate daily intervention counts.

Division 1 had the lowest MAE (6.413), while Division 3 had the highest MAE (9.716).

All six operational divisions showed negative prediction Bias. Division 6 had the strongest negative Bias (−1.732).

Annual performance varied during the validation period. The lowest MAE was observed in 2021 (7.292), while the highest was observed in 2023 (10.619).

The strongest negative annual Bias occurred in 2024 (−1.945), showing that the year with the highest forecasting error was not necessarily the year with the strongest average underestimation.

These findings show that Linear Regression forecasting performance varied across divisions and years. The results do not establish the causes of these differences.

The model was retrained monthly using expanding historical data, but prediction errors still varied over time.

These results describe Linear Regression's individual chronological validation performance. Model comparison and selection will be performed in Stage 6.


## 5.22. Build the Equal-Weight Ensemble

We will create an Ensemble using the validation predictions from LightGBM, CatBoost, and Linear Regression. Each model has already been trained with the same 48 chronological validation windows.

The Ensemble will use a **fixed equal-weight average** of the three model predictions. Each model contributes one third. These weights are defined in advance and are not fitted using validation targets, avoiding weight-selection leakage.

This is a prediction combination step, not another model training loop. The Ensemble uses only forecasts available for the corresponding division and date. The reserved 2025–2026 final test period remains excluded.


In [ ]:
# Prepare prediction columns from the three trained models
lightgbm_results = df_lightgbm_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Predicted_Count"]
].rename(columns={"Predicted_Count": "LightGBM_Prediction"}).copy()

catboost_results = df_catboost_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Predicted_Count"]
].rename(columns={"Predicted_Count": "CatBoost_Prediction"}).copy()

linear_results = df_linear_predictions[
    ["Date", "DIVISION", "Intervention_Count", "Linear_Prediction"]
].copy()

# Check that each method provides unique forecast keys
for predictions in [lightgbm_results, catboost_results, linear_results]:
    assert not predictions.duplicated(["Date", "DIVISION"]).any()
    assert len(predictions) == 8760

# Match model predictions by date and division
ensemble_results = lightgbm_results.merge(
    catboost_results, on=["Date", "DIVISION"],
    how="inner", validate="one_to_one", suffixes=("_LightGBM", "_CatBoost")
)
ensemble_results = ensemble_results.merge(
    linear_results, on=["Date", "DIVISION"],
    how="inner", validate="one_to_one"
)

# Confirm that all methods have the same target values
assert len(ensemble_results) == 8760
assert (ensemble_results["Intervention_Count_LightGBM"] ==
        ensemble_results["Intervention_Count_CatBoost"]).all()
assert (ensemble_results["Intervention_Count_LightGBM"] ==
        ensemble_results["Intervention_Count"]).all()

ensemble_results = ensemble_results.rename(
    columns={"Intervention_Count": "Actual_Count"}
)
ensemble_results = ensemble_results.drop(
    columns=["Intervention_Count_LightGBM", "Intervention_Count_CatBoost"]
)

# Use fixed, equal weights (no fitting on validation outcomes)
ensemble_model_columns = [
    "LightGBM_Prediction", "CatBoost_Prediction", "Linear_Prediction"
]
assert ensemble_results[ensemble_model_columns].notna().all().all()
ensemble_results["Ensemble_Prediction"] = (
    ensemble_results[ensemble_model_columns].mean(axis=1)
)
print("Matched observations:", len(ensemble_results))
print("Missing ensemble predictions:", ensemble_results["Ensemble_Prediction"].isna().sum())
display(ensemble_results.head())


Matched observations: 8760
Missing ensemble predictions: 0


,Date,DIVISION,LightGBM_Prediction,CatBoost_Prediction,Actual_Count,Linear_Prediction,Ensemble_Prediction
0,2021-01-01,1,23.222282,25.721262,18.0,26.262695,25.068747
1,2021-01-01,2,53.352678,54.572590,41.0,52.819793,53.581687
2,2021-01-01,3,55.816048,53.516553,48.0,51.549891,53.627497
3,2021-01-01,4,59.123836,56.256852,53.0,56.753947,57.378212
4,2021-01-01,5,57.046466,56.173112,42.0,55.034881,56.084820


## 5.23. Evaluate Ensemble Validation Performance

We will evaluate the fixed equal-weight Ensemble over the same 8,760 division-day observations from 2021 through 2024. We will calculate MAE and Bias overall, by division, and by year. Comparison and model selection will take place in Stage 6.


In [ ]:
# Calculate the Ensemble's signed forecast errors
ensemble_results["Ensemble_Error"] = (
    ensemble_results["Ensemble_Prediction"] - ensemble_results["Actual_Count"]
)

print("Ensemble Validation Results (2021-2024)")
print("Total predictions:", len(ensemble_results))
print("Overall MAE:", round(ensemble_results["Ensemble_Error"].abs().mean(), 3))
print("Overall Bias:", round(ensemble_results["Ensemble_Error"].mean(), 3))

ensemble_division_metrics = ensemble_results.groupby("DIVISION")["Ensemble_Error"].agg(
    MAE=lambda errors: errors.abs().mean(), Bias="mean"
)
print("\nPerformance by division:")
display(ensemble_division_metrics.round(3))

ensemble_results["Year"] = ensemble_results["Date"].dt.year
ensemble_yearly_metrics = ensemble_results.groupby("Year")["Ensemble_Error"].agg(
    MAE=lambda errors: errors.abs().mean(), Bias="mean"
)
print("\nPerformance by year:")
display(ensemble_yearly_metrics.round(3))


Ensemble Validation Results (2021-2024)
Total predictions: 8760
Overall MAE: 8.029
Overall Bias: -1.703

Performance by division:


,MAE,Bias
DIVISION,,
1,6.118,-1.114
2,8.136,-1.276
3,9.119,-2.880
4,8.479,-1.960
5,8.708,-1.750
6,7.617,-1.238



Performance by year:


,MAE,Bias
Year,,
2021,7.080,-0.903
2022,7.483,-1.776
2023,9.756,-2.783
2024,7.799,-1.350


**Methodological Observation**

The fixed equal-weight Ensemble was evaluated using 8,760 predictions across 48 monthly chronological validation windows from January 2021 through December 2024.

The overall MAE was 8.029 interventions, and the overall Bias was −1.703 interventions, indicating an average tendency to underestimate daily intervention counts.

The Ensemble Bias was consistent with the average Bias of the three individual models, as expected from the fixed equal-weight calculation.

Division 1 had the lowest MAE (6.118), while Division 3 had the highest MAE (9.119) and the strongest negative Bias (−2.880).

All six operational divisions showed negative prediction Bias.

Annual performance varied during the validation period. The lowest MAE was observed in 2021 (7.080), while the highest was observed in 2023 (9.756). The year 2023 also showed the strongest negative annual Bias (−2.783).

These findings show that Ensemble forecasting performance varied across divisions and years. The results do not establish the causes of these differences.

The Ensemble combines the three model predictions using fixed equal weights without additional model training or weight optimization.

These results describe the Ensemble's individual chronological validation performance. Model comparison and selection will be performed in Stage 6.

## 5.24. Save Model Training Outputs

We will save the validation predictions from four forecasting approaches: LightGBM, CatBoost, Linear Regression, and the fixed equal-weight Ensemble. All predictions are matched by date and division and include the known intervention counts.

The historical same-weekday baseline will be loaded and matched in Stage 6, which will perform the common comparison and provisional selection. The 2025–2026 final test period remains excluded.


In [ ]:
# Save only Stage 5 model outputs; baseline comparison is in Stage 6
df_model_predictions = ensemble_results[
    ["Date", "DIVISION", "Actual_Count", "LightGBM_Prediction",
     "CatBoost_Prediction", "Linear_Prediction", "Ensemble_Prediction"]
].sort_values(["Date", "DIVISION"]).reset_index(drop=True)

# Validate the exported dataset
assert len(df_model_predictions) == 8760
assert not df_model_predictions.duplicated(["Date", "DIVISION"]).any()
assert df_model_predictions.notna().all().all()
assert df_model_predictions["Date"].min() == pd.Timestamp("2021-01-01")
assert df_model_predictions["Date"].max() == pd.Timestamp("2024-12-30")

# Save model predictions for the Stage 6 comparison
predictions_file = "../../data/interim/final/SIM_Model_Validation_Predictions.csv"
df_model_predictions.to_csv(predictions_file, index=False)
print("Model predictions saved:", predictions_file)
print("Validation observations:", len(df_model_predictions))
print("Final test period excluded:", df_model_predictions["Date"].max() < pd.Timestamp("2025-01-01"))
display(df_model_predictions.head())


Model predictions saved: ../../data/interim/final/SIM_Model_Validation_Predictions.csv
Validation observations: 8760
Final test period excluded: True


,Date,DIVISION,Actual_Count,LightGBM_Prediction,CatBoost_Prediction,Linear_Prediction,Ensemble_Prediction
0,2021-01-01,1,18.0,23.222282,25.721262,26.262695,25.068747
1,2021-01-01,2,41.0,53.352678,54.572590,52.819793,53.581687
2,2021-01-01,3,48.0,55.816048,53.516553,51.549891,53.627497
3,2021-01-01,4,53.0,59.123836,56.256852,56.753947,57.378212
4,2021-01-01,5,42.0,57.046466,56.173112,55.034881,56.084820


## 5.25. Stage Conclusion — Model Training

This stage trains and evaluates LightGBM, CatBoost, and Linear Regression using 48 monthly chronological validation windows from 2021 through 2024. It also constructs and evaluates a fixed equal-weight Ensemble of the three model predictions.

All four approaches use the same division-day validation observations. Training observations follow the monthly information cutoffs, and the fitted models remain fixed within each validation month. Stage 5 exports their matched predictions to `SIM_Model_Validation_Predictions.csv`.

Stage 6 loads the historical same-weekday baseline, compares all five forecasting methods, and makes a provisional model selection. The reserved 2025–2026 final test period is not evaluated here.
